# Interpreters and Programmatic Tool Calling

An interpreter gives a Deep Agent an in-memory JavaScript workspace *inside* the agent loop: the agent writes code, the runtime runs it, and only the final result returns to the model. **Programmatic tool calling (PTC)** is what makes that workspace useful — it exposes selected tools inside the interpreter under a `tools` namespace, so the agent can call them from code.

Normally a model works one tool call at a time: call, wait, read the full result, decide the next call. Across many calls that is:

- **slow** — a model round-trip per call
- **token-expensive** — every intermediate result lands in the context window
- **unreliable at scale** — asked to hit 50 items, models tend to cover a handful

With PTC the agent writes a single script that calls tools in loops or parallel batches, filters and aggregates the results *in the interpreter*, and returns only a compact summary. The raw intermediate data never enters the model's context.

In this notebook we run the *same* web-research task two ways — plain tool calling, then PTC — and measure the difference.

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
from dotenv import load_dotenv
load_dotenv(override=True)

import time

from util import (
    count_tool_calls,
    show_run_stats,
    show_eval_code,
    show_comparison_table,
    show_comparison_bars,
)
from langchain_core.tools import tool
from tavily import TavilyClient
from deepagents import create_deep_agent
from langchain_quickjs import CodeInterpreterMiddleware
from langgraph.checkpoint.memory import MemorySaver

model = "claude-sonnet-5"

# Setting the stage

We're a specialty-coffee roastery putting together a briefing on the state of the industry in 2026. There are a dozen themes to research, and for each one we want a couple of sourced takeaways.

Each <code style="font-size:15px; color:#2b8a3e;">web_search</code> returns several results, and every result carries a title, a URL, a relevance score, and a chunk of page content — a *sizeable* payload. Twelve themes is a lot of text. Whether all of that text has to pass through the model is exactly the question PTC answers.

In [4]:
THEMES = [
    "single-origin Ethiopian coffee trends 2026",
    "cold brew coffee market growth 2026",
    "green coffee bean price outlook 2026",
    "sustainable coffee sourcing certifications 2026",
    "specialty coffee equipment innovations 2026",
    "advances in coffee decaffeination 2026",
    "coffee subscription business models 2026",
    "climate change impact on arabica coffee 2026",
    "third wave coffee shop trends 2026",
    "experimental coffee fermentation processing 2026",
    "instant specialty coffee market 2026",
    "coffee and health research 2026",
]

@tool
def web_search(query: str) -> list[dict]:
    """Search the web for a query.

    Returns a list of results, each a dict with 'title', 'url', 'content', and 'score'.
    """
    response = TavilyClient().search(query, max_results=5, search_depth="advanced")
    return response.get("results", [])

# Approach 1 — Direct Tool Calling

First, the baseline. We hand the agent the <code style="font-size:15px; color:#2b8a3e;">web_search</code> tool directly and ask for the briefing. The model calls the tool once per theme; each raw payload comes back into its context, and it reasons over all of it before writing the answer.

In [5]:
analyst_prompt = (
    "You are a market research analyst for a coffee roastery. "
    "Research every theme the user lists using the web_search tool, then write a briefing: "
    "for each theme, give 1-2 sentence takeaways followed by a source URL. Be concise."
)

request = (
    "Put together a 2026 specialty-coffee industry briefing covering these themes:\n"
    + "\n".join(f"- {t}" for t in THEMES)
)

direct_agent = create_deep_agent(
    model=model,
    tools=[web_search],
    system_prompt=analyst_prompt,
)

start = time.perf_counter()
direct_result = await direct_agent.ainvoke({"messages": [{"role": "user", "content": request}]})
direct_elapsed = time.perf_counter() - start

direct_tokens = show_run_stats(
    "Direct tool calling",
    direct_result,
    {'🔧 <code style="color:#2b8a3e;">web_search</code> calls in context': count_tool_calls(direct_result, "web_search")},
    elapsed=direct_elapsed,
)

<table style="font-size:16px; border-collapse:collapse; width:100%;"><thead><tr><th style="border:1px solid #999; padding:10px; text-align:left;">Direct tool calling</th><th style="border:1px solid #999; padding:10px; text-align:right;">Value</th></tr></thead><tbody><tr><td style="border:1px solid #999; padding:10px; text-align:left;">🔧 <code style="color:#2b8a3e;">web_search</code> calls in context</td><td style="border:1px solid #999; padding:10px; text-align:right;">12</td></tr><tr><td style="border:1px solid #999; padding:10px; text-align:left;">Input tokens</td><td style="border:1px solid #999; padding:10px; text-align:right;">56,804</td></tr><tr><td style="border:1px solid #999; padding:10px; text-align:left;">Output tokens</td><td style="border:1px solid #999; padding:10px; text-align:right;">2,947</td></tr><tr><td style="border:1px solid #999; padding:10px; text-align:left;"><b>Total tokens</b></td><td style="border:1px solid #999; padding:10px; text-align:right;"><b>59,751</b></td></tr><tr><td style="border:1px solid #999; padding:10px; text-align:left;">Wall-clock time</td><td style="border:1px solid #999; padding:10px; text-align:right;">34.5 s</td></tr></tbody></table>

# Approach 2 — Programmatic Tool Calling (PTC)

Now the same task with PTC. Two deliberate changes:

- We attach <code style="font-size:15px; color:#2b8a3e;">CodeInterpreterMiddleware</code> and put <code style="font-size:15px; color:#2b8a3e;">web_search</code> on its <code style="font-size:15px; color:#2b8a3e;">ptc</code> allowlist — and we do **not** pass it as a normal tool. The agent's only tool is now <code style="font-size:15px; color:#2b8a3e;">eval</code>; the search tool exists *only* inside the interpreter as `tools.webSearch(...)`. Raw payloads physically cannot reach the model's context.
- We raise <code style="font-size:15px; color:#2b8a3e;">timeout</code> to 120s (twelve live searches take a while) and <code style="font-size:15px; color:#2b8a3e;">max_result_chars</code> so the compact corpus returns in full.

We expect the agent to write a single script along these lines — fan out, rank, trim, then hand back only the compact corpus:

```js
const themes = [/* ... */];
const hits = await Promise.all(themes.map((q) => tools.webSearch({ query: q })));

const corpus = themes.map((theme, i) => ({
  theme,
  sources: hits[i]
    .sort((a, b) => b.score - a.score)   // best results first
    .slice(0, 3)                          // keep the top 3
    .map((r) => ({ title: r.title, url: r.url, snippet: r.content.slice(0, 300) })),
}));

corpus;   // only this compact structure returns to the model
```

We also keep a checkpointer and a fixed `thread_id` so the interpreter's state survives into the next turn.

In [1]:
ptc_prompt = (
    "You are a market research analyst for a coffee roastery with a JavaScript interpreter "
    "(the eval tool). Your web_search tool is available ONLY inside the interpreter, as "
    "tools.webSearch({query}). "
    "Research all the themes efficiently: write ONE eval script that runs every search in "
    "parallel with Promise.all, keeps the top 3 results per theme by score, trims each result's "
    "content to about 300 characters, and stores the result in a top-level variable named "
    "`corpus` (end the script with `corpus` so it is returned to you). Keep the raw search "
    "payloads inside the interpreter. Then write the briefing from `corpus`: 1-2 sentence "
    "takeaways per theme, each with a source URL."
)

ptc_agent = create_deep_agent(
    model=model,
    system_prompt=ptc_prompt,
    checkpointer=MemorySaver(),
    middleware=[
        CodeInterpreterMiddleware(
            ptc=[web_search],
            mode="thread",
            timeout=120.0,
            max_result_chars=20000,
        )
    ],
)

config = {"configurable": {"thread_id": "coffee-research"}}
start = time.perf_counter()
ptc_result = await ptc_agent.ainvoke(
    {"messages": [{"role": "user", "content": request}]},
    config=config,
)
ptc_elapsed = time.perf_counter() - start

ptc_tokens = show_run_stats(
    "Programmatic tool calling",
    ptc_result,
    {
        '🔧 <code style="color:#2b8a3e;">eval</code> calls': count_tool_calls(ptc_result, "eval"),
        '🔧 <code style="color:#2b8a3e;">web_search</code> calls in context': count_tool_calls(ptc_result, "web_search"),
    },
    elapsed=ptc_elapsed,
)

NameError: name 'create_deep_agent' is not defined

# What the agent actually wrote

The model orchestrated all twelve searches from a single `eval` call. Here's the JavaScript it produced — every raw result was ranked and trimmed *inside* the interpreter, and only the compact `corpus` came back.

In [7]:
show_eval_code(ptc_result)

<div class="highlight" style="background: #ffffff; background:#ffffff !important; color:#1a1a1a !important; border:1px solid #d0d7de; border-radius:6px; padding:12px 14px; margin:8px 0; overflow-x:auto;"><pre style="background:transparent !important; color:#1a1a1a !important; margin:0; line-height:1.45;; line-height: 125%;"><span></span><span style="color: #A90D91">const</span> <span style="color: #000">themes</span> <span style="color: #000">=</span> [
  <span style="color: #C41A16">&quot;single-origin Ethiopian coffee trends 2026&quot;</span>,
  <span style="color: #C41A16">&quot;cold brew coffee market growth 2026&quot;</span>,
  <span style="color: #C41A16">&quot;green coffee bean price outlook 2026&quot;</span>,
  <span style="color: #C41A16">&quot;sustainable coffee sourcing certifications 2026&quot;</span>,
  <span style="color: #C41A16">&quot;specialty coffee equipment innovations 2026&quot;</span>,
  <span style="color: #C41A16">&quot;advances in coffee decaffeination 2026&quot;</span>,
  <span style="color: #C41A16">&quot;coffee subscription business models 2026&quot;</span>,
  <span style="color: #C41A16">&quot;climate change impact on arabica coffee 2026&quot;</span>,
  <span style="color: #C41A16">&quot;third wave coffee shop trends 2026&quot;</span>,
  <span style="color: #C41A16">&quot;experimental coffee fermentation processing 2026&quot;</span>,
  <span style="color: #C41A16">&quot;instant specialty coffee market 2026&quot;</span>,
  <span style="color: #C41A16">&quot;coffee and health research 2026&quot;</span>
];

<span style="color: #A90D91">function</span> <span style="color: #000">trim</span>(<span style="color: #000">str</span>, <span style="color: #000">n=</span><span style="color: #1C01CE">300</span>) {
  <span style="color: #A90D91">if</span> (<span style="color: #000">!str</span>) <span style="color: #A90D91">return</span> <span style="color: #C41A16">&quot;&quot;</span>;
  <span style="color: #000">str</span> <span style="color: #000">=</span> <span style="color: #A90D91">String</span>(<span style="color: #000">str</span>);
  <span style="color: #A90D91">return</span> <span style="color: #000">str</span>.<span style="color: #000">length</span> <span style="color: #000">&gt;</span> <span style="color: #000">n</span> <span style="color: #000">?</span> <span style="color: #000">str</span>.<span style="color: #000">slice</span>(<span style="color: #1C01CE">0</span>, <span style="color: #000">n</span>) <span style="color: #000">+</span> <span style="color: #C41A16">&quot;...&quot;</span> <span style="color: #000">:</span> <span style="color: #000">str</span>;
}

<span style="color: #A90D91">const</span> <span style="color: #000">results</span> <span style="color: #000">=</span> <span style="color: #A90D91">await</span> <span style="color: #A90D91">Promise</span>.<span style="color: #000">all</span>(<span style="color: #000">themes</span>.<span style="color: #000">map</span>(<span style="color: #A90D91">async</span> (<span style="color: #000">theme</span>) =&gt; {
  <span style="color: #A90D91">const</span> <span style="color: #000">raw</span> <span style="color: #000">=</span> <span style="color: #A90D91">await</span> <span style="color: #000">tools</span>.<span style="color: #000">webSearch</span>({ <span style="color: #000">query:</span> <span style="color: #000">theme</span> });
  <span style="color: #A90D91">const</span> <span style="color: #000">arr</span> <span style="color: #000">=</span> <span style="color: #A90D91">Array</span>.<span style="color: #000">isArray</span>(<span style="color: #000">raw</span>) <span style="color: #000">?</span> <span style="color: #000">raw</span> <span style="color: #000">:</span> [];
  <span style="color: #A90D91">const</span> <span style="color: #000">sorted</span> <span style="color: #000">=</span> <span style="color: #000">arr</span>
    .<span style="color: #000">slice</span>()
    .<span style="color: #000">sort</span>((<span style="color: #000">a</span>, <span style="color: #000">b</span>) =&gt; (<span style="color: #000">b</span>.<span style="color: #000">score</span> <span style="color: #000">||</span> <span style="color: #1C01CE">0</span>) <span style="color: #000">-</span> (<span style="color: #000">a</span>.<span style="color: #000">score</span> <span style="color: #000">||</span> <span style="color: #1C01CE">0</span>))
    .<span style="color: #000">slice</span>(<span style="color: #1C01CE">0</span>, <span style="color: #1C01CE">3</span>)
    .<span style="color: #000">map</span>(<span style="color: #000">r</span> =&gt; ({
      <span style="color: #000">title:</span> <span style="color: #000">r</span>.<span style="color: #000">title</span> <span style="color: #000">||</span> <span style="color: #000">r</span>.<span style="color: #000">name</span> <span style="color: #000">||</span> <span style="color: #C41A16">&quot;&quot;</span>,
      <span style="color: #000">url:</span> <span style="color: #000">r</span>.<span style="color: #000">url</span> <span style="color: #000">||</span> <span style="color: #000">r</span>.<span style="color: #000">link</span> <span style="color: #000">||</span> <span style="color: #C41A16">&quot;&quot;</span>,
      <span style="color: #000">score:</span> <span style="color: #000">r</span>.<span style="color: #000">score</span> <span style="color: #000">||</span> <span style="color: #A90D91">null</span>,
      <span style="color: #000">content:</span> <span style="color: #000">trim</span>(<span style="color: #000">r</span>.<span style="color: #000">content</span> <span style="color: #000">||</span> <span style="color: #000">r</span>.<span style="color: #000">snippet</span> <span style="color: #000">||</span> <span style="color: #000">r</span>.<span style="color: #000">text</span> <span style="color: #000">||</span> <span style="color: #C41A16">&quot;&quot;</span>)
    }));
  <span style="color: #A90D91">return</span> { <span style="color: #000">theme</span>, <span style="color: #000">results:</span> <span style="color: #000">sorted</span> };
}));

<span style="color: #A90D91">const</span> <span style="color: #000">corpus</span> <span style="color: #000">=</span> <span style="color: #000">results</span>;
<span style="color: #000">corpus</span>;
</pre></div>


# The difference

Same task, same model, same searches — but with PTC the twelve raw payloads stayed in the interpreter and only a small corpus reached the model. That shows up directly in the token totals.

In [8]:
show_comparison_table(
    [
        ("Input tokens", direct_tokens["input"], ptc_tokens["input"]),
        ("Output tokens", direct_tokens["output"], ptc_tokens["output"]),
        ("Total tokens", direct_tokens["total"], ptc_tokens["total"]),
        ("Wall-clock time", direct_elapsed, ptc_elapsed, lambda v: f"{v:.1f} s"),
    ],
    left_label="Direct tool calling",
    right_label="Programmatic tool calling",
    emphasize=["Total tokens"],
)
show_comparison_bars(
    [
        ("Input tokens", direct_tokens["input"], ptc_tokens["input"]),
        ("Output tokens", direct_tokens["output"], ptc_tokens["output"]),
        ("Total tokens", direct_tokens["total"], ptc_tokens["total"]),
        ("Wall-clock time", direct_elapsed, ptc_elapsed, lambda v: f"{v:.1f} s"),
    ],
    left_label="Direct tool calling",
    right_label="Programmatic tool calling",
    title="Direct vs. programmatic tool calling",
)

<table style="font-size:16px; border-collapse:collapse; width:100%;"><thead><tr><th style="border:1px solid #999; padding:10px; text-align:left;">Metric</th><th style="border:1px solid #999; padding:10px; text-align:right;">Direct tool calling</th><th style="border:1px solid #999; padding:10px; text-align:right;">Programmatic tool calling</th><th style="border:1px solid #999; padding:10px; text-align:right;">Reduction</th></tr></thead><tbody><tr><td style="border:1px solid #999; padding:10px; text-align:left;">Input tokens</td><td style="border:1px solid #999; padding:10px; text-align:right;">56,804</td><td style="border:1px solid #999; padding:10px; text-align:right;">25,681</td><td style="border:1px solid #999; padding:10px; text-align:right;">55%</td></tr><tr><td style="border:1px solid #999; padding:10px; text-align:left;">Output tokens</td><td style="border:1px solid #999; padding:10px; text-align:right;">2,947</td><td style="border:1px solid #999; padding:10px; text-align:right;">2,440</td><td style="border:1px solid #999; padding:10px; text-align:right;">17%</td></tr><tr><td style="border:1px solid #999; padding:10px; text-align:left;"><b>Total tokens</b></td><td style="border:1px solid #999; padding:10px; text-align:right;"><b>59,751</b></td><td style="border:1px solid #999; padding:10px; text-align:right;"><b>28,121</b></td><td style="border:1px solid #999; padding:10px; text-align:right;"><b>53%</b></td></tr><tr><td style="border:1px solid #999; padding:10px; text-align:left;">Wall-clock time</td><td style="border:1px solid #999; padding:10px; text-align:right;">34.5 s</td><td style="border:1px solid #999; padding:10px; text-align:right;">29.9 s</td><td style="border:1px solid #999; padding:10px; text-align:right;">13%</td></tr></tbody></table>

# Summary

- **Interpreters** give a Deep Agent an in-memory JavaScript workspace; **PTC** lets the agent call your tools from that code via the `tools` namespace (`web_search` becomes `tools.webSearch`).
- Configure it with <code style="font-size:15px; color:#2b8a3e;">CodeInterpreterMiddleware(ptc=[...])</code>. Exposing a tool *only* through `ptc` (not as a normal tool) turns the interpreter into both a **context boundary** and a **permission boundary** — raw data can't leak into the model's context.
- The agent orchestrates many calls from one script — looping, batching with `Promise.all`, ranking and deduping — and returns only a compact result. Intermediate payloads stay out of context, which is where the **token savings** come from.
- Because code runs the loop, *every* item is covered — no sampling.
- With <code style="font-size:15px; color:#2b8a3e;">mode="thread"</code> (plus a checkpointer and a stable `thread_id`) the interpreter's state persists across turns, so follow-ups can reuse earlier work for free.
- PTC bridges are async — invoke with <code style="font-size:15px; color:#2b8a3e;">await agent.ainvoke(...)</code>. Interpreters are in **beta**.